In [26]:
# Dot density map de población por municipio (Tenerife). Limpia el CSV del ISTAC y dibuja el mapa.
# Genera: dot_density_tenerife.html (y poblacion_tenerife.csv con los datos ya limpios)
import glob, math, random
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go
from shapely.geometry import Point

# CONFIGURACIÓN
GEOJSON = "https://opendata.sitcan.es/upload/unidades-administrativas/gobcan_unidades-administrativas_municipios-generalizados.geojson"
CSV_ISTAC = (glob.glob("dataset-ISTAC_E30245A_000002*.csv") or ["poblacion_istac.csv"])[0]  # CSV descargado en la zona de archivos
PERSONS_PER_DOT = 500                   # 1 punto = 500 habitantes
SEED = 42
LAT_MEDIO = 28.3                        # proporción del mapa



def preparar_poblacion(ruta):
    #Esta función Limpia el CSV del ISTAC y devuelve una tabla municipio, codigo, poblacion (Tenerife, último año).
    df = pd.read_csv(ruta, dtype=str).rename(columns={
        "TERRITORIO#es": "municipio", "TERRITORIO_CODE": "codigo", "TIME_PERIOD_CODE": "anio"})
    # la medida "Población" y el total de ambos sexos
    df = df[(df["MEDIDAS_CODE"] == "POBLACION") & (df["SEXO_CODE"] == "_T")].copy()
    # último año disponible
    anio = max(df["anio"].astype(int))
    df = df[df["anio"].astype(int) == anio].copy()
    df["poblacion"] = pd.to_numeric(df["OBS_VALUE"])
    # municipios de Tenerife: aparecen entre la fila "Tenerife" (ES709) y "La Gomera" (ES706)
    orden = df["codigo"].tolist()
    ten = df.iloc[orden.index("ES709") + 1:orden.index("ES706")][["municipio", "codigo", "poblacion"]]
    ten = ten.assign(poblacion=ten["poblacion"].astype(int))
    ten.to_csv("poblacion_tenerife.csv", index=False)
    print(f"Datos limpios: año {anio}, {len(ten)} municipios, {int(ten['poblacion'].sum()):,} habitantes")
    return ten, anio



def puntos_aleatorios(geom, n, rng):
    #Muestreo por rechazo: puntos al azar en el rectángulo del municipio, solo los que caen dentro.
    minx, miny, maxx, maxy = geom.bounds
    pts, intentos = [], 0
    while len(pts) < n and intentos < n * 500 + 1000:
        p = Point(rng.uniform(minx, maxx), rng.uniform(miny, maxy))
        if geom.contains(p):
            pts.append((p.x, p.y))
        intentos += 1
    return pts


def contornos(gdf):
    #Coordenadas de los contornos de todos los municipios, en una sola línea con huecos.
    xs, ys = [], []
    for geom in gdf.geometry:
        for poly in (geom.geoms if geom.geom_type == "MultiPolygon" else [geom]):
            x, y = poly.exterior.xy
            xs += list(x) + [None]
            ys += list(y) + [None]
    return xs, ys

def main():
    pop, anio = preparar_poblacion(CSV_ISTAC)

    # mapa de municipios; se une con la población por código INE (5 dígitos)
    mapa = gpd.read_file(GEOJSON).to_crs(4326)
    mapa["codigo"] = mapa["codigo"].astype(str)
    gdf = gpd.GeoDataFrame(pop.merge(mapa[["codigo", "geometry"]], on="codigo"), crs=4326)
    assert len(gdf) == len(pop), f"Faltan municipios: {len(gdf)} de {len(pop)}"
    print("Municipios dibujados:", len(gdf), "de", len(pop))

    # puntos: población / 500 por municipio, colocados al azar dentro de su polígono
    rng = random.Random(SEED)
    px, py, nombres = [], [], []
    for _, fila in gdf.iterrows():
        n = int(round(fila["poblacion"] / PERSONS_PER_DOT))
        for (x, y) in puntos_aleatorios(fila.geometry, n, rng):
            px.append(x); py.append(y); nombres.append(fila["municipio"])
    print(f"Puntos dibujados: {len(px)}")

    cx, cy = contornos(gdf)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=cx, y=cy, mode="lines", fill="toself", fillcolor="#eceff1",
                             line=dict(color="#90a4ae", width=0.8), hoverinfo="skip", showlegend=False))
    fig.add_trace(go.Scatter(x=px, y=py, mode="markers", text=nombres,
                             marker=dict(size=4, color="#c0392b", opacity=0.7),
                             hovertemplate="%{text}<extra></extra>",
                             name=f"1 punto = {PERSONS_PER_DOT} habitantes"))

    fig.update_layout(  # aspecto general del gráfico
        title=dict(
            text=f"<b>Distribución de la Población en Tenerife</b> ({anio})", font=dict(size=20, color="#333"), x=0.5, xanchor='center'
        ),
        template="plotly_white",
        showlegend=True,
        legend=dict(
            x=0.5, y=0.02, xanchor='center', yanchor='bottom', orientation='h',bgcolor="rgba(255,255,255,0.8)", bordercolor="#eee", borderwidth=1
        ),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False, scaleanchor="x", scaleratio=1 / math.cos(math.radians(LAT_MEDIO))),
        annotations=[dict(
            text="Fuente: ISTAC y GRAFCAN vía datos.gob.es (límites municipales, no oficiales). Ubicación de puntos aleatoria por municipio.", xref="paper", yref="paper", x=0.5, y=-0.05, xanchor='center', showarrow=False, font=dict(size=12, color="#888")
        )],
        height=700, margin=dict(t=80, b=60, l=20, r=20),
    )

    fig.write_html("dot_density_tenerife.html", include_plotlyjs="cdn")
    # En Google Colab, descarga el HTML al ordenador
    try:
        from google.colab import files
        files.download("dot_density_tenerife.html")
    except ImportError:
        pass

if __name__ == "__main__":
    main()

Datos limpios: año 2025, 31 municipios, 961,745 habitantes
Municipios dibujados: 31 de 31
Puntos dibujados: 1924


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
# OHLC del IBEX 35 (últimos 3 meses). Genera: ohlc_ibex35.html
import pandas as pd
import plotly.graph_objects as go

TICKER = "^IBEX"
PERIODO = "3mo"

def descargar():
    import yfinance as yf  # librería que descarga datos de Yahoo Finance
    # una fila por día, con Open, High, Low, Close; auto_adjust=False deja los precios sin ajustar
    df = yf.download(TICKER, period=PERIODO, interval="1d", auto_adjust=False, progress=False)  # las versiones nuevas de yfinance devuelven columnas de dos niveles (campo, símbolo): nos quedamos con el campo
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df.dropna()   # elimina días con algún dato vacío

def construir(df):
    fig = go.Figure(go.Ohlc(  # crea la figura con una serie OHLC
        x=df.index, open=df["Open"], high=df["High"], low=df["Low"], close=df["Close"],
        increasing_line_color="#26a69a", decreasing_line_color="#ef5350", # Colores financieros estándar
        line_width=2,
        name="IBEX 35"
    ))

    fig.update_layout(  # aspecto general del gráfico
        title=dict(text="<b>Evolución del IBEX 35</b> (Últimos 3 meses)", font=dict(size=20, color="#333"), x=0.5),
        xaxis_title="Fecha de cotización",
        yaxis_title="Puntos del Índice",
        xaxis_rangeslider_visible=False,
        template="plotly_white",
        xaxis=dict(showgrid=False, showline=True, linewidth=1, linecolor='lightgrey'), # Ejes limpios
        yaxis=dict(showgrid=True, gridcolor='rgba(240,240,240,0.5)', showline=True, linewidth=1, linecolor='lightgrey'),
        hovermode="x unified", # Tooltip profesional que cruza toda la pantalla
        annotations=[dict(text="Fuente: Yahoo Finance (^IBEX). Verde = Alcista (Cierre > Apertura) | Rojo = Bajista (Cierre < Apertura).",
                          xref="paper", yref="paper", x=0.5, y=-0.15, showarrow=False,
                          font=dict(size=12, color="#888"))],
        height=650, margin=dict(t=80, b=80, l=60, r=40),

    )
    return fig

if __name__ == "__main__":
    df = descargar()
    if df.empty:  # si Yahoo no devuelve nada, avisa y para
        raise SystemExit("No se han descargado datos.")
    construir(df).write_html("ohlc_ibex35.html", include_plotlyjs="cdn")   # crea el HTML interactivo

    try:  # en Google Colab, descarga el HTML al ordenador
        from google.colab import files
        files.download("ohlc_ibex35.html")
    except ImportError: pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
# Convex hull sobre un scatter plot (Palmer Penguins). Genera: convex_hull_penguins.html
import seaborn as sns
import plotly.graph_objects as go
from scipy.spatial import ConvexHull          # calcula la envolvente convexa de un conjunto de puntos

# Carga el dataset y descarta los pingüinos sin alguna de las tres variables (ConvexHull falla con valores vacíos)
df = sns.load_dataset("penguins").dropna(subset=["bill_length_mm", "flipper_length_mm", "species"])
x, y = "bill_length_mm", "flipper_length_mm"  # variables de los ejes: largo del pico y largo de la aleta

colors = {"Adelie": "#4C72B0", "Chinstrap": "#C44E52", "Gentoo": "#55A868"}

def con_alfa(hex_color, alfa):
    """Convierte '#RRGGBB' en 'rgba(r,g,b,alfa)': así el relleno es transparente pero el contorno queda nítido."""
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (1, 3, 5))
    return f"rgba({r},{g},{b},{alfa})"


fig = go.Figure()

# una envolvente y una nube de puntos por especie
for sp, g in df.groupby("species"):
    pts = g[[x, y]].to_numpy()                # coordenadas (pico, aleta) de los pingüinos de esta especie

    # envolvente convexa: el polígono más pequeño que contiene todos los puntos
    h = ConvexHull(pts)
    v = list(h.vertices) + [h.vertices[0]]
    fig.add_trace(go.Scatter(
        x=pts[v, 0], y=pts[v, 1], mode="lines",               # dibuja el contorno uniendo los vértices
        fill="toself", fillcolor=con_alfa(colors[sp], 0.2),   # rellena el interior con transparencia
        line=dict(color=colors[sp], width=1.5),               # contorno opaco
        legendgroup=sp, showlegend=False,
        hoverinfo="skip"))                                    # el polígono no muestra tooltip

    # un pingüino = un punto
    fig.add_trace(go.Scatter(
        x=g[x], y=g[y], mode="markers", name=sp, legendgroup=sp,
        marker=dict(color=colors[sp], size=8, line=dict(width=0.5, color="white")),
        hovertemplate=f"<b>{sp}</b><br>Pico: %{{x}} mm<br>Aleta: %{{y}} mm<extra></extra>"))

eje = dict(showgrid=True, gridcolor="rgba(240,240,240,0.5)", showline=True, linewidth=1, linecolor="lightgrey")

fig.update_layout(
    title=dict(text="<b>Morfología de Pingüinos</b>: Envolvente convexa por especie",
               font=dict(size=20, color="#333"), x=0.5),
    xaxis_title="Longitud del pico (mm)", yaxis_title="Longitud de la aleta (mm)",
    template="plotly_white",
    xaxis=eje, yaxis=eje,
    legend=dict(title="Especies", yanchor="top", y=0.99, xanchor="left", x=0.01,
                bgcolor="rgba(255,255,255,0.8)", bordercolor="#eee", borderwidth=1),
    annotations=[dict(
        text=f"Fuente: Dataset 'Palmer Penguins' (Horst, Hill y Gorman), n = {len(df)}. El polígono delimita el clúster morfológico.",
        xref="paper", yref="paper", x=0.5, y=0, yanchor="top", yshift=-70,
        showarrow=False, font=dict(size=12, color="#888"))],
    height=650, margin=dict(t=80, b=100, l=60, r=40),
)

fig.write_html("convex_hull_penguins.html", include_plotlyjs="cdn")   # exporta el HTML interactivo

try:
    from google.colab import files
    files.download("convex_hull_penguins.html")
except ImportError:
    pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>